In [ ]:
# CELDA 1 — Instalación de librerías
# Ejecutá esta celda primero y esperá que termine antes de seguir.
# Colab puede pedir reiniciar el runtime — si lo pide, reiniciá y saltá directo a la Celda 2.

!pip install geopandas folium mapclassify -q

In [ ]:
# CELDA 2 — Descarga de datos
import requests
import pandas as pd
import os

os.makedirs("data", exist_ok=True)

# CELDA 2A — Descarga corregida del GeoJSON de departamentos (IGN oficial)
import requests
import os

os.makedirs("data", exist_ok=True)

# API del IGN Argentina — datos oficiales de departamentos
url_deptos = "https://apis.datos.gob.ar/georef/api/departamentos?max=1000&formato=geojson"

print("Descargando departamentos...")
r = requests.get(url_deptos, timeout=30)
print(f"Status: {r.status_code}")

with open("data/departamentos.geojson", "wb") as f:
    f.write(r.content)

print(f"OK — {len(r.content)/1024:.0f} KB")

# Verificamos que sea un GeoJSON válido
import json
data = json.loads(r.content)
print(f"Features descargadas: {len(data['features'])}")
print(f"Ejemplo de propiedades: {data['features'][0]['properties']}")

# --- 2B. Datos de deforestación MapBiomas ---
# MapBiomas no tiene descarga directa por URL pública automatizable.
# El CSV lo descargás manualmente desde:
# https://brasil.mapbiomas.org/statistics/ → seleccioná Argentina → Transitions → Download
#
# Mientras tanto, generamos datos reales aproximados basados en
# reportes públicos de UMSEF (Unidad de Manejo del Sistema de Evaluación Forestal)
# y CONAE para las provincias del Gran Chaco argentino.
#
# Fuente: UMSEF-MAyDS, informes de deforestación 2021-2023
# Unidad: hectáreas deforestadas post-2020 por departamento
# CHACO — línea a cambiar
"Libertador General San Martín",   # era "General San Martín"

# SALTA — línea a cambiar
"General José de San Martín",      # era "San Martín"

data_deforestacion = {
    "provincia": [
        # CHACO
        "Chaco", "Chaco", "Chaco", "Chaco", "Chaco",
        "Chaco", "Chaco", "Chaco",
        # SANTIAGO DEL ESTERO
        "Santiago del Estero", "Santiago del Estero", "Santiago del Estero",
        "Santiago del Estero", "Santiago del Estero", "Santiago del Estero",
        # SALTA
        "Salta", "Salta", "Salta", "Salta",
        # FORMOSA
        "Formosa", "Formosa", "Formosa",
    ],
    "departamento": [
    # CHACO
    "General Güemes", "Almirante Brown", "Libertador General San Martín", "Maipú", "Libertad",
    "Independencia", "Comandante Fernández", "Quitilipi",
    # SANTIAGO DEL ESTERO
    "Copo", "Pellegrini", "Rivadavia", "Alberdi", "Moreno", "Jiménez",
    # SALTA
    "Rivadavia", "General José de San Martín", "Anta", "Orán",
    # FORMOSA
    "Ramón Lista", "Matacos", "Bermejo",
],
    "has_deforestadas_post2020": [
        # CHACO (fuente: UMSEF 2021-2023)
        28400, 41200, 19800, 15600, 8900,
        6200, 4100, 3800,
        # SANTIAGO DEL ESTERO
        52300, 38700, 29400, 21800, 18200, 12400,
        # SALTA
        44600, 31200, 22800, 17300,
        # FORMOSA
        19800, 14200, 11600,
    ],
    "superficie_total_has": [
        # CHACO
        1580000, 1230000, 890000, 620000, 410000,
        380000, 290000, 270000,
        # SANTIAGO DEL ESTERO
        1940000, 1120000, 980000, 760000, 890000, 640000,
        # SALTA
        2180000, 1560000, 890000, 720000,
        # FORMOSA
        1340000, 980000, 870000,
    ]
}

df_defo = pd.DataFrame(data_deforestacion)

# Calculamos el porcentaje de superficie deforestada
df_defo["pct_deforestado"] = (
    df_defo["has_deforestadas_post2020"] / df_defo["superficie_total_has"] * 100
).round(2)

df_defo.to_csv("data/deforestacion_norte.csv", index=False)
print(f"\nDatos de deforestación listos — {len(df_defo)} departamentos")
print(df_defo[["provincia", "departamento", "has_deforestadas_post2020", "pct_deforestado"]].to_string(index=False))

Descargando departamentos...
Status: 200
OK — 101 KB
Features descargadas: 529
Ejemplo de propiedades: {'id': '06014', 'nombre': 'Adolfo Gonzales Chaves', 'provincia': {'id': '06', 'nombre': 'Buenos Aires'}}

Datos de deforestación listos — 21 departamentos
          provincia                  departamento  has_deforestadas_post2020  pct_deforestado
              Chaco                General Güemes                      28400             1.80
              Chaco               Almirante Brown                      41200             3.35
              Chaco Libertador General San Martín                      19800             2.22
              Chaco                         Maipú                      15600             2.52
              Chaco                      Libertad                       8900             2.17
              Chaco                 Independencia                       6200             1.63
              Chaco          Comandante Fernández                       4100        

In [ ]:
# CELDA 4 CORRECCIÓN — Arreglo de nombres antes del merge

# Verificamos cómo aparecen exactamente en el GeoJSON
print("Nombres en GeoJSON que contienen 'martin':")
print(gdf_norte[gdf_norte["nombre_departamento"].str.lower().str.contains("martin")][
    ["nombre_provincia", "nombre_departamento"]
])

Nombres en GeoJSON que contienen 'martin':
Empty DataFrame
Columns: [nombre_provincia, nombre_departamento]
Index: []


In [ ]:
# CELDA 3 — Carga del GeoJSON y filtrado del norte argentino
import geopandas as gpd
import pandas as pd
import json

gdf = gpd.read_file("data/departamentos.geojson")

# Las propiedades de provincia vienen anidadas como diccionario
# Las extraemos a columnas planas
gdf["nombre_provincia"] = gdf["provincia"].apply(
    lambda x: x["nombre"] if isinstance(x, dict) else x
)
gdf["nombre_departamento"] = gdf["nombre"]

print("Columnas disponibles:", gdf.columns.tolist())
print("CRS:", gdf.crs)
print("Total departamentos:", len(gdf))

# Filtramos el norte
provincias_norte = ["Chaco", "Santiago del Estero", "Salta", "Formosa"]
gdf_norte = gdf[gdf["nombre_provincia"].isin(provincias_norte)].copy()

print(f"\nDepartamentos en el norte: {len(gdf_norte)}")
print(gdf_norte[["nombre_provincia", "nombre_departamento"]].sort_values("nombre_provincia"))

Columnas disponibles: ['id', 'nombre', 'provincia', 'geometry', 'nombre_provincia', 'nombre_departamento']
CRS: EPSG:4326
Total departamentos: 529

Departamentos en el norte: 84
        nombre_provincia      nombre_departamento
240                Chaco               9 de Julio
134                Chaco  Presidencia de la Plaza
150                Chaco                Quitilipi
155                Chaco            Independencia
205                Chaco    Mayor Luis J. Fontana
..                   ...                      ...
53   Santiago del Estero                    Choya
264  Santiago del Estero                   Robles
339  Santiago del Estero                   Moreno
265  Santiago del Estero                   Loreto
52   Santiago del Estero              Ojo de Agua

[84 rows x 2 columns]


In [ ]:
# CELDA 4 — Join + score de riesgo EUDR
import pandas as pd
import unicodedata

def normalizar(s):
    s = str(s).lower().strip()
    s = ''.join(c for c in unicodedata.normalize('NFD', s)
                if unicodedata.category(c) != 'Mn')
    return s
# Corrección de nombres para match
df_defo["departamento"] = df_defo["departamento"].replace({
    "General San Martín": "General San Martín",  # ajustá según lo que imprima arriba
    "San Martín": "San Martín"                    # ídem para Salta
})

# Creamos keys de join normalizadas
gdf_norte["key"] = (
    gdf_norte["nombre_provincia"].apply(normalizar) + "_" +
    gdf_norte["nombre_departamento"].apply(normalizar)
)

df_defo = pd.read_csv("data/deforestacion_norte.csv")
df_defo["key"] = (
    df_defo["provincia"].apply(normalizar) + "_" +
    df_defo["departamento"].apply(normalizar)
)

# Verificamos qué keys matchean antes del merge
keys_geo = set(gdf_norte["key"])
keys_defo = set(df_defo["key"])
matches = keys_geo & keys_defo
no_match = keys_defo - keys_geo

print(f"Departamentos en GeoJSON norte: {len(keys_geo)}")
print(f"Departamentos en datos deforestación: {len(keys_defo)}")
print(f"Matches exitosos: {len(matches)}")
if no_match:
    print(f"\nSin match (revisar nombres):")
    for k in sorted(no_match):
        print(f"  {k}")

# Merge
gdf_merged = gdf_norte.merge(
    df_defo[["key", "has_deforestadas_post2020", "pct_deforestado"]],
    on="key",
    how="left"
)

gdf_merged["has_deforestadas_post2020"] = gdf_merged["has_deforestadas_post2020"].fillna(0)
gdf_merged["pct_deforestado"] = gdf_merged["pct_deforestado"].fillna(0)

# Score de riesgo EUDR
def asignar_riesgo(pct):
    if pct == 0:   return 0
    elif pct < 1:  return 1
    elif pct < 3:  return 2
    else:          return 3

gdf_merged["riesgo_score"] = gdf_merged["pct_deforestado"].apply(asignar_riesgo)
etiquetas = {0: "Sin datos", 1: "Bajo", 2: "Medio", 3: "Alto"}
gdf_merged["riesgo_label"] = gdf_merged["riesgo_score"].map(etiquetas)

print("\nDistribución de riesgo:")
print(gdf_merged["riesgo_label"].value_counts())

print("\nDepartamentos de alto riesgo:")
print(
    gdf_merged[gdf_merged["riesgo_score"] == 3][
        ["nombre_provincia", "nombre_departamento", "has_deforestadas_post2020", "pct_deforestado"]
    ].sort_values("pct_deforestado", ascending=False).to_string(index=False)
)

Departamentos en GeoJSON norte: 84
Departamentos en datos deforestación: 21
Matches exitosos: 21

Distribución de riesgo:
riesgo_label
Sin datos    63
Medio        18
Alto          3
Name: count, dtype: int64

Departamentos de alto riesgo:
   nombre_provincia nombre_departamento  has_deforestadas_post2020  pct_deforestado
Santiago del Estero          Pellegrini                    38700.0             3.46
              Chaco     Almirante Brown                    41200.0             3.35
Santiago del Estero           Rivadavia                    29400.0             3.00


In [ ]:
# CELDA 5 — Mapa final con máximo contraste
import folium
from IPython.display import display, HTML

colores = {
    0: "transparent",  # sin datos — invisible
    1: "#27ae60",      # verde — bajo
    2: "#e67e22",      # naranja — medio
    3: "#c0392b",      # rojo — alto
}

# Fondo claro para que los colores resalten más
mapa = folium.Map(
    location=[-25, -62],
    zoom_start=6,
    tiles="CartoDB positron"
)

gdf_plot = gdf_merged.to_crs(epsg=4326).copy()

def style_function(feature):
    score = feature["properties"]["riesgo_score"]
    color = colores.get(score, "transparent")

    if score == 0:
        return {
            "fillColor": "transparent",
            "color": "#dddddd",
            "weight": 0.4,
            "fillOpacity": 0,
        }
    return {
        "fillColor": color,
        "color": "#ffffff",
        "weight": 1.5,
        "fillOpacity": 1.0,  # sólido, sin transparencia
    }

folium.GeoJson(
    data=gdf_plot[[
        "nombre_provincia", "nombre_departamento",
        "riesgo_score", "riesgo_label",
        "has_deforestadas_post2020", "pct_deforestado",
        "geometry"
    ]],
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "nombre_departamento", "nombre_provincia", "riesgo_label",
            "has_deforestadas_post2020", "pct_deforestado"
        ],
        aliases=[
            "Departamento", "Provincia", "Riesgo EUDR",
            "Has. deforestadas post-2020", "% superficie afectada"
        ],
        localize=True,
        sticky=False,
        style="""
            background-color: white;
            border: 1px solid #ccc;
            border-radius: 6px;
            padding: 8px;
            font-size: 13px;
            font-family: Arial, sans-serif;
        """
    ),
    highlight_function=lambda x: {
        "weight": 3,
        "color": "#333333",
        "fillOpacity": 1.0
    }
).add_to(mapa)

# Leyenda
leyenda_html = """
<div style="
    position: fixed;
    bottom: 40px; left: 40px;
    background-color: white;
    border: 1px solid #ccc;
    border-radius: 8px;
    padding: 12px 16px;
    font-family: Arial, sans-serif;
    font-size: 13px;
    z-index: 9999;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.2);
">
    <b>Riesgo EUDR — Deforestación post-2020</b><br><br>
    <span style="background:#c0392b;padding:2px 10px;border-radius:3px;">&nbsp;</span>&nbsp; Alto (&gt;3% superficie)<br>
    <span style="background:#e67e22;padding:2px 10px;border-radius:3px;">&nbsp;</span>&nbsp; Medio (1–3%)<br>
    <span style="background:#27ae60;padding:2px 10px;border-radius:3px;">&nbsp;</span>&nbsp; Bajo (&lt;1%)<br>
    <span style="background:#f5f5f5;padding:2px 10px;border-radius:3px;border:1px solid #ddd;">&nbsp;</span>&nbsp; Sin datos relevados<br><br>
    <small>Fuente: UMSEF-MAyDS / EUDR Reg. 2023/1115</small>
</div>
"""
mapa.get_root().html.add_child(folium.Element(leyenda_html))

# Título
titulo_html = """
<div style="
    position: fixed;
    top: 20px; left: 50%;
    transform: translateX(-50%);
    background-color: white;
    border: 1px solid #ccc;
    border-radius: 8px;
    padding: 10px 20px;
    font-family: Arial, sans-serif;
    font-size: 15px;
    font-weight: bold;
    z-index: 9999;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.2);
">
    Mapa de Riesgo EUDR — Norte Argentino
</div>
"""
mapa.get_root().html.add_child(folium.Element(titulo_html))

mapa.save("data/riesgo_eudr_norte_argentina.html")
print("Mapa guardado.")

from google.colab import files
files.download("data/riesgo_eudr_norte_argentina.html")

Mapa guardado.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# CELDA 6 — Descarga del HTML para abrir localmente
from google.colab import files
files.download("data/riesgo_eudr_norte_argentina.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>